In [ ]:
%matplotlib ipympl
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import altair as alt
import ipywidgets as widgets
import matplotlib.pyplot as plt


In [ ]:
orbits = pl.read_parquet("outputs/orbits.parquet")
orbits

In [ ]:
orbit_data = pl.read_parquet("outputs/orbit_data.parquet")
orbit_data.sort("Ph", descending=True)

In [ ]:
custom_colors = {
    0: "#BAC24C",  # yellow
    1: "#5179D6",  # orange
    2: "#2ca02c",  # green
    3: "#d62728",  # red
    # 4: "#9467bd",  # purple
    5: "#9651D6",  # brown
}

chart3 = (
    alt.Chart(
        orbit_data.with_columns((pl.col("Ph") * 1000.0).alias("Ph_scaled"))
    )
    .mark_circle(size=100)
    .encode(
        x=alt.X("fd:Q").title("Drive Frequency [Hz]"),
        y=alt.Y("Ph_scaled").scale(type="log").title("Harvested Power [mW]"),
        #y=alt.Y("Ph_scaled").title("Harvested Power [mW]"),
        color=alt.Color(
            "detected_subharmonic:N",
            title="Detected subharmonic",
            scale=alt.Scale(
                domain=list(custom_colors.keys()),
                range=list(custom_colors.values()),
            ),
        ),
        shape="detected_subharmonic:N",
        tooltip=["orbit_label", "fd", "Ph", "detected_subharmonic"],
    )
    .properties(
        title="Damping dissipation vs. Drive Frequency", width=800, height=400
    )
).interactive()

chart3.show()

In [ ]:
fig = go.Figure()
scat0 = px.scatter_3d(
    orbits,
    x="x",
    z="dotx",
    y="fd",
    color="detected_subharmonic",
    #size=(orbits["subharmonic"] + 1.0) ** -1 * 0 + 0.05,
    labels={
        "x": "Position, x",
        "dotx": r"Speed, dot x",
        "fd": "Driving frequency, f_d [Hz]",
    },
)
fig.add_trace(scat0.data[0])
fig.update_traces(marker=dict(size=3))

fig.show()



In [ ]:
custom_colors = {
    0: "#BAC24C",  # yellow
    1: "#5179D6",  # orange
    2: "#2ca02c",  # green
    3: "#d62728",  # red
    # 4: "#9467bd",  # purple
    5: "#9651D6",  # brown
}

fd = 50.0
x = "x"
y = "dotx"
xlabel = "Position, x"
ylabel = "Speed, dot x"

orbits_fd = orbits.filter(pl.col("fd") == fd)
orbits_data_fd = orbit_data.filter(pl.col("fd") == fd)
orbits_dic = {}
for row in orbits_fd.iter_rows(named=True):
    olabel = row["orbit_label"]
    alabel = row["attractor_label"]
    if olabel not in orbits_dic.keys():
        orbits_dic[olabel] = {"orbit":{}, "data":row}
    odata = pl.read_parquet(f"outputs/orbits_from_attractors/orbit_{olabel}_attractor_{alabel}.parquet")
    orbits_dic[olabel]["orbit"][alabel] = odata

fig = plt.figure(figsize=(10, 8))
for olabel, odata in orbits_dic.items():
    alabel = odata["data"]["attractor_label"]
    detected_subharmonic = odata["data"]["detected_subharmonic"]
    for alabel, orbit in odata["orbit"].items():
        plt.plot(
            orbit[x],
            orbit[y],
            color=custom_colors[detected_subharmonic],
            label=f"Orbit {olabel} (subharmonic {detected_subharmonic})",
        )
        plt.plot(
            orbit[x][0:1],
            orbit[y][0:1],
            color=custom_colors[detected_subharmonic],
            marker="o",
        )
#plt.legend()
plt.grid()
plt.title(f"Orbits at Drive Frequency fd={fd} Hz")
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.show()


In [ ]:
odata["orbit"]